# SahyogX: Exploratory Data Analysis & Welfare Risk Modeling
### AI-Based Predictive Personnel Stress and Welfare Monitoring System for Uniformed Forces

> **DISCLAIMER**: Synthetic data is used for prototype/testing purposes and does not represent actual personnel.
> This system predicts operational welfare/stress indicators for proactive leadership check-ins; it does **not** diagnose clinical or psychiatric conditions.

## 1. Research Context: Public Datasets vs. Operational Realities

In uniformed forces (defense, paramilitary, police), operational logs, deployment durations, tactical duty hours, and leave schedules are strictly classified or controlled under **Operational Security (OPSEC)**.

Publicly accessible defense health research:
- **Army STARRS / STARRS-LS (ICPSR 35198/36340)**: Provides survey distributions and risk/resilience factors, but tactical deployment telemetry is redacted under restricted enclaves.
- **DoD Health Related Behaviors Survey (HRBS by RAND/DoD)**: Reports cross-sectional lifestyle, sleep deficit, and duty strain profiles without unit movement logs.
- **OPM Federal Viewpoint Survey (FEVS)**: Measures civilian workload and burnout, but lacks operational field/combat indicators.

Hence, SahyogX utilizes a mathematically grounded synthetic generator informed by published military wellness findings.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Load synthetic dataset
data_path = "../data/raw/personnel_welfare_synthetic_raw.csv"
df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]} records, {df.shape[1]} columns")
df.head()

## 2. Target Distribution & Class Imbalance

Uniformed personnel stress naturally follows an imbalanced distribution: most personnel maintain operational readiness (`LOW`), a subset experiences strain (`MODERATE`), and a critical minority requires immediate outreach (`ELEVATED`).

In [ ]:
class_counts = df["risk_category"].value_counts()
class_proportions = df["risk_category"].value_counts(normalize=True)
pd.DataFrame({"Count": class_counts, "Proportion": class_proportions.round(4)})

## 3. Operational Feature Statistics by Risk Tier

Inspect key behavioral indicators (duty hours, sleep hours, deployment duration, leave lag) across risk tiers.

In [ ]:
summary_cols = [
    "duty_hours_weekly",
    "avg_sleep_hours",
    "deployment_duration_months",
    "days_since_last_leave",
    "peer_support_score",
    "recovery_rest_days_monthly",
]
df.groupby("risk_category")[summary_cols].mean().round(2)

## 4. Feature Engineering Verification

We compute composite metrics: sleep deficit, cumulative fatigue index, operational strain index, and leave deprivation ratio.

In [ ]:
import sys
sys.path.append("../..")
from ml.src.feature_engineering import add_engineered_features

df_engineered = add_engineered_features(df)
engineered_cols = [
    "sleep_deficit_hours",
    "composite_fatigue_index",
    "deployment_to_recovery_ratio",
    "leave_deprivation_index",
    "operational_strain_index",
    "protective_buffer_score",
    "net_vulnerability_index"
]
df_engineered[engineered_cols].describe().round(2)

## 5. Model Inference & Explainability Demonstration

Test the production `WelfareRiskPredictor` interface on an operational sample.

In [ ]:
from ml.src.predict import WelfareRiskPredictor

predictor = WelfareRiskPredictor.from_directory("../models")

sample_soldier = {
    "personnel_id": "PX-EX-01",
    "unit_type": "Infantry",
    "role_operational_intensity": "Extreme",
    "duty_hours_weekly": 72.0,
    "overtime_hours_weekly": 16.0,
    "deployment_duration_months": 10.5,
    "deployments_last_3_years": 3,
    "days_since_last_leave": 180,
    "leave_days_taken_annual": 15.0,
    "recovery_rest_days_monthly": 1.0,
    "avg_sleep_hours": 4.2,
    "sleep_disruption_index": 8.0,
    "physical_readiness_score": 68.0,
    "wellness_survey_score": 11.0,
    "peer_support_score": 3.5,
    "environmental_hardship_score": 4.5,
}

prediction = predictor.predict(sample_soldier)
import json
print(json.dumps(prediction, indent=2))
print("\nExplainability indicators:")
for indicator in prediction["contributing_indicators"]:
    print(f"- {indicator}")